In [ ]:
#| default_exp foundation

## Foundation

Small shared mechanics: CLI behavior, failure telemetry, cell parsing, validation, chapter navigation, and dynamic cell classification for notebook context.

This notebook is the shared basement of the project. It keeps the small private helpers that every public tool relies on: CLI behavior, safe error reporting, cell selection, cell parsing, semantic cell classification, and chapter addressing.

Most functions here are intentionally private. The rest of the project can stay user-facing because this notebook absorbs the messy details of notebooks as data structures.

Read it as a tour of the shapes nbskill needs to recognize. A notebook is not just text: it has cells, headings, generated Python files, visible outputs, and tiny bits of metadata. The foundation layer gives all later notebooks one vocabulary for those shapes, so the public tools can talk about notebooks without making users think in raw JSON.

### Production contract

This notebook owns the shared invariants for every other tool. Production behavior is: path helpers resolve notebooks from project-root and notebook-dir execution, cell-block parsing creates valid notebook cells, metadata stamping records semantic classes and export hashes, and validation reports missing or stale metadata before edits proceed.


The helpers here deliberately stay small because every higher-level tool depends on them. For example, `parse_cells` turns friendly cell-block text into notebook cells, the semantic classifiers let readers say "show me tests" or "show me exported code" without parsing raw notebook JSON.

In [ ]:
from contextlib import redirect_stdout
from io import StringIO
import os


In [ ]:
#| export
import ast,hashlib,json,os,re
import shutil,subprocess,sys
from contextlib import contextmanager
from pathlib import Path
from fastcore.basics import in_jupyter
from fastcore.nbio import mk_cell, new_nb, write_nb
from fastcore.script import _in_call_parse


### Demo scratch files

Examples and tests in later notebooks need small notebooks they can safely mutate. These helpers keep that setup in one place: they create named artifacts under nbs/data, reset them before use, and remove them when the example is finished.

In [ ]:
#| export
def remove_demo_path(path):
    path = Path(path)
    if path.is_dir(): shutil.rmtree(path)
    elif path.exists(): path.unlink()
    return path

`demo_path` names a scratch artifact and optionally removes any previous copy. It only creates the parent folder; helpers that write content decide what belongs at the path.

In [ ]:
#| export
def demo_path(name, base="nbs/data", reset=True):
    path = Path(base) / name
    if reset: remove_demo_path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    return path

In [ ]:
#| export
def path_candidates(path):
    "Return cwd-relative path candidates for root and notebook-dir execution."
    pth = Path(str(path)).expanduser()
    if pth.is_absolute(): return [pth]
    parts = pth.parts
    in_notebook_dir = parts and parts[0] == Path.cwd().name and (in_jupyter() or Path.cwd().name == "nbs")
    if not in_notebook_dir: return [pth]
    stripped = Path(*parts[1:]) if len(parts) > 1 else Path(".")
    return [pth, stripped]

In [ ]:
_path_candidate_rows = path_candidates("nbs/00_foundation.ipynb")
assert _path_candidate_rows[0] == Path("nbs/00_foundation.ipynb")
if Path.cwd().name == "nbs":
    assert _path_candidate_rows[-1] == Path("00_foundation.ipynb")
else:
    assert _path_candidate_rows == [Path("nbs/00_foundation.ipynb")]
assert path_candidates(Path("/tmp/example.ipynb")) == [Path("/tmp/example.ipynb")]
print("path_candidates handles root and notebook-dir paths")

Notebook helpers should fail gently when a path is missing, unreadable, or shaped like ordinary JSON instead of a Jupyter notebook. `is_valid_ipynb` keeps that contract small and dependency-free.

In [ ]:
#| export
_VALID_CELL_TYPES = {"code", "markdown", "raw"}


def _is_valid_notebook_cell(cell):
    if not isinstance(cell, dict): return False
    if cell.get("cell_type") not in _VALID_CELL_TYPES: return False
    if not isinstance(cell.get("metadata"), dict): return False
    source = cell.get("source")
    if not isinstance(source, (str, list)): return False
    if isinstance(source, list) and not all(isinstance(line, str) for line in source): return False
    if cell["cell_type"] == "code":
        if "outputs" not in cell or not isinstance(cell.get("outputs"), list): return False
        execution_count = cell.get("execution_count")
        if execution_count is not None and not isinstance(execution_count, int): return False
    return True

In [ ]:
#| export
def is_valid_ipynb(path):
    try:
        data = json.loads(Path(path).read_text(encoding="utf-8"))
    except (OSError, UnicodeDecodeError, json.JSONDecodeError):
        return False
    if not isinstance(data, dict): return False
    if not isinstance(data.get("nbformat"), int): return False
    if not isinstance(data.get("nbformat_minor"), int): return False
    if not isinstance(data.get("metadata"), dict): return False
    cells = data.get("cells")
    return isinstance(cells, list) and all(_is_valid_notebook_cell(cell) for cell in cells)

`write_demo_notebook` owns the whole lifetime of a scratch notebook: create it, yield its path to the example, then remove it even if the example fails.

In [ ]:
#| export
@contextmanager
def write_demo_notebook(name, cells=None, base="nbs/data", reset=True):
    path = demo_path(name, base=base, reset=reset)
    cells = cells or [
        mk_cell("## Demo notebook\nThis tiny notebook gives nbskill tools something real to inspect.", cell_type="markdown"),
        mk_cell("#| export\ndef demo_answer():\n    return 42"),
        mk_cell("assert demo_answer() == 42")]
    nb = new_nb(cells)
    write_nb(nb, path)
    try:
        yield path
    finally:
        remove_demo_path(path)

Here is the tiny notebook fixture in the wild. It gives examples a real `.ipynb` to read, then vanishes after the context exits, which keeps the repo tidy while still letting the rendered notebook show concrete output.

In [ ]:
from fastcore.nbio import read_nb

with write_demo_notebook("00_foundation_tour.ipynb") as path:
    nb = read_nb(path)
    print(path.name)
    print([cell.cell_type for cell in nb.cells])
    print(str(nb.cells[0].source).splitlines()[0])

print(path.exists())

In [ ]:
with write_demo_notebook("00_foundation_bad.ipynb") as bad_path:
    bad_path.write_text("{}", encoding="utf-8")
    assert not is_valid_ipynb(bad_path)
    assert not is_valid_ipynb(bad_path.with_suffix(".missing.ipynb"))

### CLI contracts and diagnostics

The public functions in later notebooks are both Python functions and command-line commands. These helpers make that dual use predictable: direct Python calls return values, CLI calls print user-friendly errors, and tool starts or failures are recorded in a small local failure map for debugging repeated friction.

In [ ]:
#| export
def cli_return(value=None):
    return None if _in_call_parse.get() else value

In [ ]:
#| export
def cli_error(msg):
    if _in_call_parse.get():
        print(msg, file=sys.stderr)
        raise SystemExit(1)
    raise ValueError(msg)

In [ ]:
#| export
def failure_map_path():
    default = Path.home() / ".nbskill-errors.json"
    return Path(os.environ.get("NBSKILL_FAILURE_MAP", default)).expanduser()


In [ ]:
#| export
def empty_failure_map():
    return {"version": 1, "events": [], "counts": {}, "last_call": None}


In [ ]:
#| export
def load_failure_map(path):
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError, OSError):
        data = empty_failure_map()
    data.setdefault("version", 1)
    data.setdefault("events", [])
    data.setdefault("counts", {})
    data.setdefault("last_call", None)
    return data

In [ ]:
#| export
_NBSKILL_HOOKS_MARKER_START = "# nbskill nbdev hooks:start"


In [ ]:
#| export
_NBSKILL_HOOKS_MARKER_END = "# nbskill nbdev hooks:end"


In [ ]:
#| export
_NBSKILL_HOOKS_INSTALLED_SENTINEL = "nbskill-hooks-installed"


In [ ]:
#| export
_NBSKILL_HOOKS_DISABLED_SENTINEL = "nbskill-hooks-disabled"


In [ ]:
#| export
def git_run(root, *args):
    "Run a git command from `root` and return stdout, or empty text on failure."
    proc = subprocess.run(["git", "-C", str(root), *args], text=True, capture_output=True)
    return proc.stdout if proc.returncode == 0 else ""


def git_root(path="."):
    "Return the containing git repository root, if one exists."
    path = Path(path).expanduser()
    base = path.parent if path.suffix else path
    candidates = [base, *path_candidates(base), Path(".")]
    for candidate in dict.fromkeys(candidates):
        out = git_run(candidate, "rev-parse", "--show-toplevel").strip()
        if out: return Path(out)
    return None

In [ ]:
#| export
def git_status_paths(root):
    "Return paths with working-tree changes relative to `root`."
    paths = set()
    for line in git_run(root, "status", "--porcelain").splitlines():
        raw = line[3:].strip() if len(line) > 3 else line.strip()
        if " -> " in raw: raw = raw.split(" -> ", 1)[1]
        if raw: paths.add(raw)
    return paths


def git_tracked_paths(root):
    "Return git-tracked paths relative to `root`."
    return set(git_run(root, "ls-files").splitlines())

In [ ]:
#| export
def git_diff_stats(root, ref="HEAD"):
    "Return added/deleted line stats for working-tree changes from `ref`."
    stats = {}
    for line in git_run(root, "diff", "--numstat", ref, "--").splitlines():
        added, deleted, path = line.split("\t", 2)
        stats[path] = {
            "added": int(added) if added.isdigit() else 0,
            "deleted": int(deleted) if deleted.isdigit() else 0,
        }
    return stats

In [ ]:
#| export
def _is_notebook_path(path):
    path = Path(path)
    hidden = any(part.startswith(".") or part == ".ipynb_checkpoints" for part in path.parts)
    return path.suffix == ".ipynb" and not hidden and not path.name.startswith("_")


def _has_glob_chars(path):
    return any(char in str(path) for char in "*?[]")

In [ ]:
#| export
def _glob_notebook_paths(path):
    try:
        from fastcore.xtras import globtastic
        if _has_glob_chars(path):
            import glob
            return [Path(item) for item in glob.glob(str(path), recursive=True)]
        return [Path(item) for item in globtastic(path, file_glob="*.ipynb", skip_folder_re=r"(^[_.]|\.ipynb_checkpoints)", skip_file_re=r"^[_.]", func=Path)]
    except Exception:
        return sorted(Path(path).rglob("*.ipynb")) if Path(path).is_dir() else []


def _nbdev_notebook_paths(path):
    try:
        from nbdev.doclinks import nbglob
        return [Path(item) for item in nbglob(path=path, as_path=True)]
    except Exception:
        return []

In [ ]:
#| export
def notebook_paths(path="nbs"):
    "Return visible notebook paths for a file, directory, glob, or nbdev project path."
    raw = "." if path is None else str(path)
    for candidate in path_candidates(raw):
        pth = Path(candidate).expanduser()
        if pth.is_file():
            paths = [pth]
        elif _has_glob_chars(raw):
            paths = _glob_notebook_paths(pth)
        elif pth.is_dir():
            paths = _nbdev_notebook_paths(pth) or _glob_notebook_paths(pth)
        else:
            paths = []
        paths = sorted({item for item in paths if _is_notebook_path(item)})
        if paths: return paths
    return []

In [ ]:
#| export
def source_without_directives(source):
    "Return source with nbdev cell directives removed."
    return "\n".join(line for line in str(source or "").splitlines() if not line.lstrip().startswith("#|"))

In [ ]:
#| export
def file_line_count(path):
    "Return the number of text lines in `path`, or zero when it cannot be read."
    try: return len(Path(path).read_text(encoding="utf-8", errors="ignore").splitlines())
    except OSError: return 0


def cap_text(text, limit=2000, max_output_chars=None):
    "Cap text, returning a string or review-style metadata when `max_output_chars` is used."
    metadata = max_output_chars is not None
    limit = max_output_chars if metadata else limit
    text = "" if text is None else str(text)
    if limit is None or len(text) <= limit:
        capped, omitted = text, 0
    else:
        omitted = len(text) - limit
        capped = f"{text[:limit].rstrip()}\n... truncated {omitted} chars ..."
    if not metadata: return capped
    return {"text": capped, "truncated": bool(omitted), "chars": len(text), "omitted_chars": omitted}

In [ ]:
#| export
_GENERATED_HEADER_RE = re.compile(r"^# AUTOGENERATED! DO NOT EDIT! File to edit: (.+)$")


def _generated_owner_from_header(path):
    try: lines = Path(path).read_text(encoding="utf-8", errors="ignore").splitlines()[:3]
    except OSError: return None
    for line in lines:
        match = _GENERATED_HEADER_RE.match(line.strip())
        if match: return (Path(path).parent / match.group(1).rstrip(".")).resolve()
    return None


def generated_owner(path, notebooks=None):
    "Return the source notebook for an nbdev-generated Python file, when known."
    path = Path(path)
    if path.suffix != ".py" or not path.exists(): return None
    owner = _generated_owner_from_header(path)
    if owner is not None or notebooks is None: return owner
    target = path.resolve()
    for nb_path in notebook_paths(notebooks):
        try: py_path = exported_py_path(nb_path)
        except (FileNotFoundError, OSError): py_path = None
        if py_path is not None and Path(py_path).exists() and Path(py_path).resolve() == target:
            return Path(nb_path).resolve()
    return None

In [ ]:
#| export
def _looks_like_nbdev_project(root):
    root = Path(root)
    pyproject = root / "pyproject.toml"
    if (root / "nbs").exists() or (root / "settings.ini").exists(): return True
    return pyproject.exists() and "[tool.nbdev]" in pyproject.read_text(encoding="utf-8", errors="ignore")


In [ ]:
#| export
def _nbdev_hook_block():
    return f"""{_NBSKILL_HOOKS_MARKER_START}
run_nbdev_cmd() {{
  if command -v "$1" >/dev/null 2>&1; then
    "$@"
  elif command -v uv >/dev/null 2>&1; then
    uv run "$@"
  else
    echo "nbskill: missing $1; install nbdev or uv" >&2
    exit 127
  fi
}}

run_nbdev_cmd nbdev-clean
if ! git diff --quiet -- .; then
  echo "nbskill: nbdev-clean changed notebooks. Review and stage those changes before committing." >&2
  exit 1
fi
run_nbdev_cmd nbdev-test
{_NBSKILL_HOOKS_MARKER_END}
"""


In [ ]:
#| export
def _replace_marked_block(text, block):
    if _NBSKILL_HOOKS_MARKER_START in text and _NBSKILL_HOOKS_MARKER_END in text:
        before = text.split(_NBSKILL_HOOKS_MARKER_START, 1)[0].rstrip()
        after = text.split(_NBSKILL_HOOKS_MARKER_END, 1)[1].lstrip()
        return f"{before}\n\n{block}\n{after}".rstrip() + "\n"
    prefix = text.rstrip() if text.strip() else "#!/bin/sh"
    return f"{prefix}\n\n{block}\n"


In [ ]:
#| export
def _hook_state_paths(root):
    info = Path(root) / ".git" / "info"
    return info / _NBSKILL_HOOKS_INSTALLED_SENTINEL, info / _NBSKILL_HOOKS_DISABLED_SENTINEL


In [ ]:
#| export
def _has_nbskill_hook_block(pre_commit):
    if not pre_commit.exists(): return False
    text = pre_commit.read_text(encoding="utf-8", errors="ignore")
    return _NBSKILL_HOOKS_MARKER_START in text and _NBSKILL_HOOKS_MARKER_END in text


In [ ]:
#| export
def _hooks_removed_by_user(root, pre_commit):
    installed, disabled = _hook_state_paths(root)
    if disabled.exists(): return True
    if installed.exists() and not _has_nbskill_hook_block(pre_commit):
        disabled.write_text("nbskill hooks were removed; not reinstalling automatically\n", encoding="utf-8")
        return True
    return False


In [ ]:
#| export
def install_nbdev_pre_commit_hooks(path=".", run_nbdev_install_hooks=True):
    "Install nbdev-clean and nbdev-test pre-commit hooks in a git-backed nbdev project."
    root = git_root(path)
    if root is None: return {"installed": False, "reason": "not-a-git-repo"}
    if not _looks_like_nbdev_project(root): return {"installed": False, "reason": "not-an-nbdev-project", "root": str(root)}
    hooks = root / ".git" / "hooks"
    pre_commit = hooks / "pre-commit"
    if _hooks_removed_by_user(root, pre_commit): return {"installed": False, "reason": "hooks-removed-by-user", "root": str(root)}
    if run_nbdev_install_hooks:
        cmd = ["nbdev-install-hooks"] if shutil.which("nbdev-install-hooks") else None
        if cmd is None and shutil.which("uv"): cmd = ["uv", "run", "nbdev-install-hooks"]
        if cmd is not None: subprocess.run(cmd, cwd=root, text=True, capture_output=True)
    hooks.mkdir(parents=True, exist_ok=True)
    text = pre_commit.read_text(encoding="utf-8", errors="ignore") if pre_commit.exists() else ""
    pre_commit.write_text(_replace_marked_block(text, _nbdev_hook_block()), encoding="utf-8")
    pre_commit.chmod(pre_commit.stat().st_mode | 0o111)
    installed, _ = _hook_state_paths(root)
    installed.parent.mkdir(parents=True, exist_ok=True)
    installed.write_text("nbskill hooks installed once\n", encoding="utf-8")
    return {"installed": True, "root": str(root), "hook": str(pre_commit)}

### Parsing user-facing selectors

Selectors are the boundary between human shorthand and list operations. A caller might ask for `3`, `2:5`, `-1`, or `None`; these helpers normalize that text before any read or write tool touches notebook cells. Keeping the grammar here makes higher-level tools predictable and keeps cell-index mistakes from spreading.

In [ ]:
#| export
def parse_literal(value):
    if value is None: return None
    if isinstance(value, str):
        value = value.strip()
        if value.lower() in {"", "none", "null"}: return None
        try: return ast.literal_eval(value)
        except (SyntaxError, ValueError): return value
    return value

In [ ]:
#| export
def none_if_string(value):
    return None if isinstance(value, str) and value.strip().lower() in {"", "none", "null"} else value

In [ ]:
#| export
def _parse_slice(value):
    if not isinstance(value, str) or ":" not in value: return None
    parts = value.split(":")
    if len(parts) not in (2, 3): return None
    vals = [int(p) if p else None for p in parts]
    return slice(*vals)

In [ ]:
#| export
def _as_index(value, length):
    idx = int(value)
    if idx < 0: idx += length
    if idx < 0 or idx >= length: raise IndexError(value)
    return idx

In [ ]:
#| export
def _parse_read_selector(value):
    value = parse_literal(value)
    if value is None: return None
    if isinstance(value, str):
        slc = _parse_slice(value)
        if slc is not None: return slc
        return int(value)
    if isinstance(value, (list, tuple)): return [int(o) for o in value]
    return int(value)

In [ ]:
#| export
def _parse_write_target(value):
    value = parse_literal(value)
    if value is None: return None
    if isinstance(value, str):
        slc = _parse_slice(value)
        if slc is not None: return slc
        return int(value)
    if isinstance(value, (list, tuple)):
        if len(value) != 2: raise ValueError("write ranges must have start and stop")
        return slice(value[0], value[1])
    return int(value)

In [ ]:
#| export
def _select_cells(cells, selector):
    items = list(enumerate(cells))
    if selector is None: return items
    selector = _parse_read_selector(selector)
    if isinstance(selector, slice): return items[selector]
    if isinstance(selector, list):
        return [(idx, cells[idx]) for idx in (_as_index(o, len(cells)) for o in selector)]
    idx = _as_index(selector, len(cells))
    return [(idx, cells[idx])]

In [ ]:
#| export
def _delete_cells(cells, selector):
    if selector is None: return
    target = _parse_write_target(selector)
    if isinstance(target, slice):
        del cells[target]
        return
    idx = _as_index(target, len(cells))
    del cells[idx]

### Turning text into notebook cells

Write tools accept plain text because that is what agents and humans can produce quickly. This section turns that text into real cells: split on block separators, honor `%%markdown`/`%%code` markers, and split exported code into symbol-sized chunks when needed. It is the small parser that makes notebook edits feel like editing a document instead of assembling JSON.

In [ ]:
#| export
def _split_blocks(text):
    text = "" if text is None else str(text)
    if not text: return []
    return [o.strip("\n") for o in re.split(r"(?m)^\s*---\s*$", text) if o.strip()]

In [ ]:
#| export
def _coerce_cell(cell, default_type="code"):
    if isinstance(cell, dict): return cell
    if isinstance(cell, (tuple, list)) and len(cell) == 2:
        cell_type, source = cell
        return mk_cell(str(source), cell_type=str(cell_type))
    return mk_cell(str(cell), cell_type=default_type)

In [ ]:
#| export
def is_definition_node(node):
    return isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef))

In [ ]:
#| export
def node_start_line(node):
    return min([node.lineno, *[d.lineno for d in getattr(node, "decorator_list", [])]]) - 1

In [ ]:
#| export
def is_export_directive(line):
    return re.match(r"^\s*#\|\s*(export|exports|exporti)(\s|$)", line) is not None

In [ ]:
#| export
def _is_export_gap(lines):
    return bool(lines) and all((not line.strip()) or is_export_directive(line) for line in lines)

In [ ]:
#| export
def _emit_code_chunk(chunks, lines, export_prefix=None):
    if export_prefix: lines = [*export_prefix, *lines]
    text = "\n".join(lines).strip("\n")
    if text: chunks.append(text)

In [ ]:
#| export
def _split_code_cell_sources(source):
    source = source.strip("\n")
    if not source: return []
    try: tree = ast.parse(source)
    except SyntaxError: return [source]
    if sum(1 for node in tree.body if is_definition_node(node)) <= 1: return [source]

    lines = source.splitlines()
    first_start = node_start_line(tree.body[0]) if tree.body else 0
    leading = lines[:first_start]
    shared_export = [line for line in leading if is_export_directive(line)] if _is_export_gap(leading) else []
    chunks = []
    if shared_export:
        cursor = first_start
    else:
        _emit_code_chunk(chunks, leading)
        cursor = first_start

    for node in tree.body:
        start = node_start_line(node)
        end = node.end_lineno
        gap = lines[cursor:start]
        if shared_export and _is_export_gap(gap): gap = []
        _emit_code_chunk(chunks, [*gap, *lines[start:end]], shared_export or None)
        cursor = end
    _emit_code_chunk(chunks, lines[cursor:])
    return chunks or [source]

In [ ]:
#| export
def _split_code_cell(cell):
    cell_type = cell.get("cell_type") if isinstance(cell, dict) else getattr(cell, "cell_type", None)
    if cell_type != "code": return [cell]
    source = cell.get("source", "") if isinstance(cell, dict) else getattr(cell, "source", "")
    if isinstance(source, list): source = "".join(source)
    sources = _split_code_cell_sources(str(source))
    if len(sources) <= 1: return [cell]
    return [mk_cell(source, cell_type="code") for source in sources]

In [ ]:
#| export
def _split_symbol_cells(cells):
    split = []
    for cell in cells: split.extend(_split_code_cell(cell))
    return split

In [ ]:
#| export
def cell_source(cell):
    source = cell.get("source", "") if isinstance(cell, dict) else getattr(cell, "source", "")
    if isinstance(source, list): return "".join(source)
    return str(source)

### Cell metadata and semantic classes

Once cells can be parsed, the tools need names for what they *are doing*. A code cell with output is an example; a code cell with an `assert` is a test; a markdown cell with prose is documentation. Those labels power filters such as `type=tests` and help review tools talk about notebooks in human terms.

In [ ]:
#| export
_NBSKILL_METADATA_KEY = "nbskill"


In [ ]:
#| export
def cell_metadata(cell):
    meta = cell.get("metadata", None) if isinstance(cell, dict) else getattr(cell, "metadata", None)
    if meta is None:
        meta = {}
        if isinstance(cell, dict): cell["metadata"] = meta
        else: cell.metadata = meta
    return meta


In [ ]:
#| export
def notebook_metadata(nb):
    meta = nb.get("metadata", None) if isinstance(nb, dict) else getattr(nb, "metadata", None)
    if meta is None:
        meta = {}
        if isinstance(nb, dict): nb["metadata"] = meta
        else: nb.metadata = meta
    return meta


In [ ]:
#| export
def _nbskill_cell_metadata(cell, create=True):
    meta = cell_metadata(cell) if create else (cell.get("metadata", {}) if isinstance(cell, dict) else getattr(cell, "metadata", {}) or {})
    info = meta.get(_NBSKILL_METADATA_KEY) if isinstance(meta, dict) else None
    if isinstance(info, dict): return info
    if not create: return None
    info = {}
    meta[_NBSKILL_METADATA_KEY] = info
    return info


In [ ]:
#| export
def _nbskill_notebook_metadata(nb, create=True):
    meta = notebook_metadata(nb) if create else (nb.get("metadata", {}) if isinstance(nb, dict) else getattr(nb, "metadata", {}) or {})
    info = meta.get(_NBSKILL_METADATA_KEY) if isinstance(meta, dict) else None
    if isinstance(info, dict): return info
    if not create: return None
    info = {}
    meta[_NBSKILL_METADATA_KEY] = info
    return info


In [ ]:
#| export
def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


In [ ]:
#| export
def _metadata_path(path):
    path = Path(path)
    try: return path.resolve().relative_to(Path.cwd().resolve()).as_posix()
    except (OSError, ValueError): return path.as_posix()


In [ ]:
#| export
def _default_exp_from_notebook(nb):
    for cell in getattr(nb, "cells", []):
        for line in cell_source(cell).splitlines():
            match = re.match(r"^\s*#\|\s*default_exp\s+(.+?)\s*$", line)
            if match: return match.group(1).strip()
    return None


In [ ]:
#| export
def exported_py_path(nb_path, nb=None):
    "Return the generated Python file path for an nbdev notebook, if it has one."
    nb_path = Path(nb_path)
    if nb is None:
        from fastcore.nbio import read_nb
        nb = read_nb(nb_path)
    default_exp = _default_exp_from_notebook(nb)
    if not default_exp: return None
    try:
        from nbdev.config import get_config
        lib_path = Path(get_config(nb_path.parent).lib_path)
    except Exception:
        lib_path = nb_path.parent.parent / default_exp.split(".", 1)[0]
    return lib_path / (default_exp.replace(".", "/") + ".py")


In [ ]:
assert source_without_directives("#| export\nx = 1\n#| default_exp demo") == "x = 1"
with write_demo_notebook("foundation_notebook_paths.ipynb") as nb_path:
    checkpoint = nb_path.parent / ".ipynb_checkpoints" / "skip.ipynb"
    checkpoint.parent.mkdir(parents=True, exist_ok=True)
    write_nb(new_nb([mk_cell("assert True")]), checkpoint)
    try:
        paths = notebook_paths(nb_path.parent)
        assert nb_path in paths
        assert checkpoint not in paths
    finally:
        remove_demo_path(checkpoint.parent)
generated = demo_path("generated_owner_sample.py")
try:
    generated.write_text("# AUTOGENERATED! DO NOT EDIT! File to edit: ../00_foundation.ipynb.\n", encoding="utf-8")
    assert generated_owner(generated).name == "00_foundation.ipynb"
    assert file_line_count(generated) == 1
finally:
    remove_demo_path(generated)
assert cap_text("abcdef", limit=3).startswith("abc")
assert cap_text("abcdef", max_output_chars=3)["truncated"] is True

In [ ]:
#| export
def stamp_export_metadata(nb, py_path):
    info = _nbskill_notebook_metadata(nb)
    info["exported_py_path"] = _metadata_path(py_path)
    info["exported_py_hash"] = file_hash(py_path)
    return nb


In [ ]:
#| export
def _fresh_semantic_metadata(cell):
    info = _nbskill_cell_metadata(cell, create=False)
    if not info: return None
    if info.get("cell_type") != getattr(cell, "cell_type", None): return None
    types = info.get("semantic_types")
    if not isinstance(types, list): return None
    normalized = tuple("example_cell" if str(item) == "exploration_cell" else str(item) for item in types)
    if getattr(cell, "cell_type", None) != "code" and "unclean_cell" in normalized: return None
    semantic = tuple(name for name in normalized if name not in {"exported_code", "unclean_cell"})
    if "unclean_cell" in normalized and len(semantic) <= 1: return None
    return normalized


In [ ]:
#| export
def parse_one_cell(text, default_type="code"):
    if isinstance(text, (list, tuple)):
        if len(text) != 1:
            cli_error("update_cell expects exactly one replacement cell; use write_nb or batch_edit_nb for multi-cell edits")
        return _coerce_cell(text[0], default_type)
    blocks = _split_blocks(text)
    if len(blocks) != 1:
        cli_error("update_cell expects exactly one replacement cell; remove standalone '---' separators or use write_nb/batch_edit_nb for multi-cell edits")
    return _cell_from_block(blocks[0], default_type)

In [ ]:
#| export
def find_cell_by_id(cells, cell_id):
    matches = [(idx, cell) for idx, cell in enumerate(cells) if getattr(cell, "id", None) == cell_id]
    if len(matches) == 1: return matches[0]
    if not matches: cli_error(f"No cell has id {cell_id!r}")
    cli_error(f"Multiple cells have id {cell_id!r}")

In [ ]:
#| export
def find_cell_by_text(cells, old_str):
    matches = [(idx, cell) for idx, cell in enumerate(cells) if old_str in cell_source(cell)]
    if len(matches) == 1: return matches[0]
    if not matches: cli_error("old_str did not match any cell")
    idxs = ", ".join(str(idx) for idx, _ in matches)
    cli_error(f"old_str matched multiple cells: {idxs}. Use --cell_id or a more specific old_str.")

In [ ]:
#| export
def replace_cell(nb, idx, new_cell):
    old_id = getattr(nb.cells[idx], "id", None)
    if old_id is not None: new_cell.id = old_id
    nb.cells[idx] = new_cell

In [ ]:
#| export
def clear_outputs(cell):
    if getattr(cell, "cell_type", None) == "code":
        cell.outputs = []
        cell.execution_count = None
    return cell

In [ ]:
#| export
def _looks_like_multiline_cli_text(text):
    if not isinstance(text, str) or "\\n" not in text: return False
    stripped = text.lstrip().lower()
    if stripped.startswith(("%%code\\n", "%%markdown\\n", "%%md\\n", "%%raw\\n")): return True
    if "\\n---\\n" in text: return True
    return "\\n    " in text or "\\n\t" in text


In [ ]:
#| export
def _decode_cli_newlines(text):
    return text.replace("\\n", "\n") if _looks_like_multiline_cli_text(text) else text


In [ ]:
#| export
def load_cells_text(cells="", cells_file=None, decode_newlines=True):
    if cells_file:
        if cells: raise ValueError("Use either cells or cells_file, not both")
        return Path(cells_file).expanduser().read_text(encoding="utf-8")
    if cells == "-": return sys.stdin.read()
    return _decode_cli_newlines(cells) if decode_newlines else cells


In [ ]:
#| export
def _should_validate_python(source):
    for line in source.splitlines():
        stripped = line.lstrip()
        if stripped.startswith(("%", "!")): return False
    return bool(source.strip())

In [ ]:
#| export
def _format_syntax_error(source, err, cell_idx):
    lines = source.splitlines()
    line = lines[err.lineno - 1] if err.lineno and 0 < err.lineno <= len(lines) else ""
    pointer = " " * max((err.offset or 1) - 1, 0) + "^" if line else ""
    msg = [f"Invalid Python in new code cell {cell_idx}: {err.msg} at line {err.lineno}, column {err.offset}"]
    if line: msg += [line, pointer]
    msg.append("Tip: shell quoting can turn backslash-n escapes into real newlines inside Python strings. Use --cells_file PATH or cells=- for complex code.")
    return chr(10).join(msg)

In [ ]:
#| export
def validate_code_cells(cells):
    for idx, cell in enumerate(cells):
        cell_type = cell.get("cell_type") if isinstance(cell, dict) else getattr(cell, "cell_type", None)
        if cell_type != "code": continue
        source = cell_source(cell)
        if not _should_validate_python(source): continue
        try: ast.parse(source)
        except SyntaxError as err:
            msg = _format_syntax_error(source, err, idx)
            if _in_call_parse.get(): raise SystemExit(msg)
            raise ValueError(msg) from err

In [ ]:
#| export
def _cell_from_block(block, default_type="code"):
    lines = block.splitlines()
    marker = lines[0].strip().lower() if lines else ""
    cell_type = default_type
    if marker in {"%%markdown", "%%md"}: cell_type, lines = "markdown", lines[1:]
    elif marker == "%%code": cell_type, lines = "code", lines[1:]
    elif marker == "%%raw": cell_type, lines = "raw", lines[1:]
    return mk_cell("\n".join(lines), cell_type=cell_type)


In [ ]:
#| export
def parse_cells(cells, default_type="code"):
    if isinstance(cells, (list, tuple)): return _split_symbol_cells([_coerce_cell(o, default_type) for o in cells])

    parsed = [_cell_from_block(block, default_type) for block in _split_blocks(cells)]
    return _split_symbol_cells(parsed)


### Naming cells by behavior

The reading and MCP layers present cells by meaning, not just by `code` or `markdown`. This section detects imports, private helpers, exported code, tests, examples, docs, section headers, and mixed cells so callers can filter notebooks at a useful level.

In [ ]:
#| export
def first_line(source):
    for line in source.splitlines():
        line = line.strip()
        if line: return line
    return ""

In [ ]:
#| export
def cell_prefix(idx, cell, show_ids=False): return f"Cell id={cell.id}: {cell.cell_type}"

In [ ]:
#| export
def _format_chapter_spans(spans, cells):
    lines = []
    for span in spans:
        cell = cells[span["start"]]
        lines.append(f"Chapter id={cell.id}: ## {span['title']}")
    return "\n".join(lines)

In [ ]:
#| export
def matches_filter(source, pattern):
    pattern = str(pattern)
    if pattern in source: return True
    try: return re.search(pattern, source, flags=re.MULTILINE) is not None
    except re.error: return False

In [ ]:
#| export
def is_exported_code_cell(cell):
    if getattr(cell, "cell_type", None) != "code": return False
    return any(is_export_directive(line) for line in cell_source(cell).splitlines())


In [ ]:
#| export
def _cell_outputs(cell): return list(getattr(cell, "outputs", []) or [])


In [ ]:
#| export
def _has_cell_output(cell): return bool(_cell_outputs(cell))


In [ ]:
#| export
def _code_tree(cell):
    try: return ast.parse(cell_source(cell))
    except SyntaxError: return None


In [ ]:
#| export
def _code_body(cell):
    tree = _code_tree(cell)
    return [] if tree is None else list(tree.body)


In [ ]:
#| export
def _is_import_cell(cell):
    if getattr(cell, "cell_type", None) != "code": return False
    body = _code_body(cell)
    return bool(body) and all(isinstance(node, (ast.Import, ast.ImportFrom)) for node in body)


In [ ]:
#| export
def _has_private_function(cell):
    if getattr(cell, "cell_type", None) != "code": return False
    tree = _code_tree(cell)
    if tree is None: return False
    return any(
        isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name.startswith("_")
        for node in tree.body)


In [ ]:
#| export
def _has_test_marker(cell):
    source = cell_source(cell)
    tree = _code_tree(cell)
    if tree is None: return re.search(r"\b(assert|test_[A-Za-z0-9_]*)\b", source) is not None
    if any(isinstance(node, ast.Assert) for node in ast.walk(tree)): return True
    return any(
        isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name.startswith("test_")
        for node in tree.body)


In [ ]:
#| export
def _is_test_cell(cell):
    return getattr(cell, "cell_type", None) == "code" and not _has_cell_output(cell) and _has_test_marker(cell)


In [ ]:
#| export
def _is_example_cell(cell):
    return getattr(cell, "cell_type", None) == "code" and _has_cell_output(cell)


In [ ]:
#| export
def _is_section_header(cell):
    if getattr(cell, "cell_type", None) != "markdown": return False
    return any(re.match(r"^#{1,2}\s+", line.strip()) for line in cell_source(cell).splitlines())


In [ ]:
#| export
def _is_docs_cell(cell):
    if getattr(cell, "cell_type", None) != "markdown": return False
    return any(line.strip() and not re.match(r"^#{1,2}\s+", line.strip()) for line in cell_source(cell).splitlines())


In [ ]:
#| export
def _cell_base_class_names(cell):
    names = []
    if _is_import_cell(cell): names.append("import_cell")
    if _is_example_cell(cell): names.append("example_cell")
    if _is_test_cell(cell): names.append("test_cell")
    if _has_private_function(cell): names.append("private_code")
    if is_exported_code_cell(cell): names.append("exported_code")
    if _is_docs_cell(cell): names.append("docs_cell")
    if _is_section_header(cell): names.append("section_header")
    return tuple(names)


In [ ]:
#| export
def _semantic_code_class_names(cell):
    semantic = {"import_cell", "example_cell", "test_cell", "private_code"}
    return tuple(name for name in _cell_base_class_names(cell) if name in semantic)


In [ ]:
#| export
def _fallback_cell_class_name(cell):
    cell_type = getattr(cell, "cell_type", None)
    return f"{cell_type}_cell" if cell_type else "unknown_cell"


In [ ]:
#| export
def _computed_cell_class_names(cell):
    names = _cell_base_class_names(cell)
    if getattr(cell, "cell_type", None) == "code" and len(_semantic_code_class_names(cell)) > 1:
        return (*names, "unclean_cell")
    return names or (_fallback_cell_class_name(cell),)


In [ ]:
#| export
def _refresh_cell_metadata(cell):
    info = _nbskill_cell_metadata(cell)
    semantic_types = _computed_cell_class_names(cell)
    info["cell_type"] = getattr(cell, "cell_type", None)
    info["semantic_types"] = list(semantic_types)
    info.pop("source_hash", None)
    return cell


In [ ]:
#| export
def stamp_notebook_metadata(nb, exported_py_path=None):
    for cell in getattr(nb, "cells", []): _refresh_cell_metadata(cell)
    if exported_py_path is not None: stamp_export_metadata(nb, exported_py_path)
    return nb


In [ ]:
#| export
def cell_class_names(cell): return _fresh_semantic_metadata(cell) or _computed_cell_class_names(cell)


In [ ]:
with write_demo_notebook("00_foundation_context.ipynb") as path:
    assert path.exists()
    assert is_valid_ipynb(path)
assert not path.exists()

`parse_cells` is the little translator between a friendly edit format and real notebook cells. The dashed line means "start a new cell", while `%%markdown` and `%%code` make the intended cell type explicit.

In [ ]:
_demo_text = "\n".join(["%%markdown", "## Demo", "---", "%%code", "value = 42"])
_demo_cells = parse_cells(_demo_text)
for idx, cell in enumerate(_demo_cells):
    print(cell_prefix(idx, cell))
    print("  type:", cell.cell_type)
    print("  first line:", first_line(cell_source(cell)))

In [ ]:
#| export
def _normalize_cell_type_filter(value):
    if value is None: return None
    aliases = {
        "code": "code",
        "py": "code",
        "python": "code",
        "md": "markdown",
        "markdown": "markdown",
        "doc": "markdown",
        "docs": "markdown",
        "raw": "raw",
        "export": "exported_code",
        "exported": "exported_code",
        "exported_code": "exported_code",
        "import": "import_cell",
        "imports": "import_cell",
        "import_cell": "import_cell",
        "private": "private_code",
        "private_code": "private_code",
        "test": "test_cell",
        "tests": "test_cell",
        "test_cell": "test_cell",
        "example": "example_cell",
        "examples": "example_cell",
        "example_cell": "example_cell",
        "exploration": "example_cell",
        "explorations": "example_cell",
        "exploration_cell": "example_cell",
        "docs_cell": "docs_cell",
        "documentation": "docs_cell",
        "section": "section_header",
        "header": "section_header",
        "section_header": "section_header",
        "unclean": "unclean_cell",
        "unclean_cell": "unclean_cell",
    }
    normalized = set()
    for item in str(value).split(","):
        key = item.strip().lower()
        if not key: continue
        if key not in aliases:
            choices = ", ".join(sorted(set(aliases)))
            raise ValueError(f"Unknown cell_type {item!r}; use one of: {choices}")
        normalized.add(aliases[key])
    return normalized or None


In [ ]:
#| export
def cell_matches_type(cell, cell_type):
    wanted = _normalize_cell_type_filter(cell_type)
    if wanted is None: return True
    if getattr(cell, "cell_type", None) in wanted: return True
    return bool(set(cell_class_names(cell)) & wanted)


In [ ]:
#| export
def with_context(cells, items, include=False):
    if not include: return items

    idxs = {idx for idx, _ in items}
    for idx in list(idxs):
        prev = idx - 1
        while prev >= 0 and cells[prev].cell_type == "markdown":
            idxs.add(prev)
            prev -= 1

        nxt = idx + 1
        while nxt < len(cells):
            cell = cells[nxt]
            if cell.cell_type != "code" or is_exported_code_cell(cell): break
            idxs.add(nxt)
            nxt += 1
    return [(idx, cells[idx]) for idx in sorted(idxs)]


### Chapters as editing scopes

Chapters are the notebook-native version of a module section. A heading cell opens a span, and everything until the next heading belongs to that span. Write, execute, and read tools use these spans so users can say "the query language chapter" instead of counting cell indexes by hand.

In [ ]:
#| export
def _chapter_title(cell):
    if getattr(cell, "cell_type", None) != "markdown": return None
    for line in cell_source(cell).splitlines():
        match = re.match(r"^##\s+(.+?)\s*$", line.strip())
        if match: return match.group(1).strip()
    return None

In [ ]:
#| export
def _chapter_spans(cells):
    starts = [(idx, title) for idx, cell in enumerate(cells) if (title := _chapter_title(cell))]
    spans = []
    for pos, (start, title) in enumerate(starts):
        end = starts[pos + 1][0] if pos + 1 < len(starts) else len(cells)
        spans.append(dict(title=title, start=start, end=end))
    return spans

In [ ]:
#| export
def _matching_chapters(cells, chapter=None):
    spans = _chapter_spans(cells)
    if chapter is None: return spans
    return [span for span in spans if matches_filter(span["title"], chapter)]

In [ ]:
#| export
def chapter_index_set(cells, chapter):
    idxs = set()
    for span in _matching_chapters(cells, chapter):
        idxs.update(range(span["start"], span["end"]))
    return idxs

In [ ]:
#| export
def one_chapter(cells, chapter, create=False):
    matches = _matching_chapters(cells, chapter)
    if len(matches) == 1: return matches[0]
    if not matches and create:
        cells.append(mk_cell(f"## {chapter}", cell_type="markdown"))
        return dict(title=str(chapter), start=len(cells) - 1, end=len(cells))
    if not matches: raise ValueError(f"No chapter matches {chapter!r}")
    titles = ", ".join(f"{span['title']} ({span['start']}:{span['end']})" for span in matches)
    raise ValueError(f"Chapter {chapter!r} matches multiple chapters: {titles}")

On a real notebook, chapter spans become a tiny table of contents with cell ranges. Here `index.ipynb` is being parsed by the foundation helpers that the reading tools themselves rely on.

In [ ]:
_index_cells = read_nb("index.ipynb").cells
for span in _chapter_spans(_index_cells)[:5]: print(f"{span['title']}: cells {span['start']}..{span['end'] - 1}")

print("reading indexes:", sorted(chapter_index_set(_index_cells, "Reading"))[:5])

In [ ]:
#| export
def _chapter_body_len(span):
    return max(span["end"] - span["start"] - 1, 0)

In [ ]:
#| export
def _chapter_body_slice(span, target):
    body_start = span["start"] + 1
    body_len = _chapter_body_len(span)
    start, stop, step = target.indices(body_len)
    if step != 1: raise ValueError("chapter ranges do not support steps")
    return slice(body_start + start, body_start + stop)

In [ ]:
#| export
def _chapter_delete(cells, span, selector):
    if selector is None: return
    target = _parse_write_target(selector)
    body_start = span["start"] + 1
    body_len = _chapter_body_len(span)
    if isinstance(target, slice):
        del cells[_chapter_body_slice(span, target)]
        return
    idx = int(target)
    if idx < 0: idx += body_len
    if idx < 0 or idx >= body_len: raise IndexError(target)
    del cells[body_start + idx]

In [ ]:
#| export
def _chapter_insert_target(span, target):
    body_start = span["start"] + 1
    body_len = _chapter_body_len(span)
    if target is None: return slice(body_start, body_start + body_len)
    if isinstance(target, slice): return _chapter_body_slice(span, target)
    idx = int(target)
    if idx == -1: return body_start + body_len
    if idx < 0: idx += body_len
    if idx < 0 or idx > body_len: raise IndexError(target)
    return body_start + idx

In [ ]:
custom_map = demo_path("00_foundation_errors.json")
_old_failure_map = os.environ.get("NBSKILL_FAILURE_MAP")
try:
    os.environ["NBSKILL_FAILURE_MAP"] = str(custom_map)
    assert failure_map_path() == custom_map
finally:
    if _old_failure_map is None: os.environ.pop("NBSKILL_FAILURE_MAP", None)
    else: os.environ["NBSKILL_FAILURE_MAP"] = _old_failure_map

In [ ]:

cells = parse_cells("%%markdown\n## Demo\n---\n%%code\nvalue = 42")
assert len(cells) == 2
assert cells[0].cell_type == "markdown"
assert cells[1].source == "value = 42"
one = parse_one_cell("%%code\nvalue = 99")
assert one.cell_type == "code"
assert one.source == "value = 99"
assert load_cells_text("%%code\\nvalue = 1") == "%%code\nvalue = 1"
assert load_cells_text("def f():\\n    return 1") == "def f():\n    return 1"
assert load_cells_text("value = 'a\\nb'") == "value = 'a\\nb'"
encoded = "def f():" + chr(92) + "n    return 1"
assert load_cells_text(encoded, decode_newlines=False) == encoded


Here are the labels on toy cells. This is deliberately small, but it is the same classification that lets the reading and review tools distinguish examples from tests without asking a model to guess.

In [ ]:
_semantic_samples = {}
_semantic_samples["import"] = mk_cell("import os", cell_type="code")
_semantic_samples["private"] = mk_cell("def _helper():\n    pass", cell_type="code")
_semantic_samples["exported"] = mk_cell("#| export\ndef public():\n    pass", cell_type="code")
_semantic_samples["test"] = mk_cell("assert 1 == 1", cell_type="code")
_semantic_samples["plain"] = mk_cell("value = 1", cell_type="code")
_semantic_samples["docs"] = mk_cell("Some notes", cell_type="markdown")
_semantic_samples["example"] = mk_cell("print('hello notebook')", cell_type="code")
_semantic_samples["example"].outputs = [dict(output_type="stream", name="stdout", text="hello notebook\n")]

for label, cell in _semantic_samples.items(): print(f"{label:8} -> {', '.join(cell_class_names(cell))}")

The assertions below are the boring-but-important contract. They pin down the edge cases: mixed header/prose markdown, exported import cells that are normal nbdev cells, true mixed semantic code, and export metadata that points back to the generated `.py` file.

In [ ]:
import_cell = _semantic_samples["import"]
private_cell = _semantic_samples["private"]
exported_cell = _semantic_samples["exported"]
test_cell = _semantic_samples["test"]
plain_code_cell = _semantic_samples["plain"]
example_cell = _semantic_samples["example"]
docs_cell = _semantic_samples["docs"]

expected_classes = [
    (import_cell, ("import_cell",)), (private_cell, ("private_code",)), (exported_cell, ("exported_code",)),
    (test_cell, ("test_cell",)), (plain_code_cell, ("code_cell",)), (example_cell, ("example_cell",)),
    (docs_cell, ("docs_cell",))]
for cell, expected in expected_classes: assert cell_class_names(cell) == expected

section_cell = mk_cell("## API", cell_type="markdown")
docs_header_cell = mk_cell("## API\nSome notes", cell_type="markdown")
exported_import_cell = mk_cell("#| export\nimport os", cell_type="code")
unclean_cell = mk_cell("assert True\ndef _helper():\n    return 1", cell_type="code")
assert cell_class_names(section_cell) == ("section_header",)
assert cell_class_names(docs_header_cell) == ("docs_cell", "section_header")
assert cell_class_names(exported_import_cell) == ("import_cell", "exported_code")
assert cell_class_names(unclean_cell) == ("test_cell", "private_code", "unclean_cell")
assert cell_matches_type(example_cell, "example")
assert cell_matches_type(example_cell, "exploration")

Metadata stamping is separate from classification: it records the generated Python artifact and caches the semantic labels that downstream tools can reuse.

In [ ]:
export_path = demo_path("00_foundation_export.py")
export_path.write_text("print('exported')\n", encoding="utf-8")
stamped_nb = stamp_notebook_metadata(new_nb([exported_cell]), exported_py_path=export_path)
stamped_cell = stamped_nb.cells[0]
cell_info = stamped_cell.metadata["nbskill"]
notebook_info = notebook_metadata(stamped_nb)["nbskill"]
assert cell_info["cell_type"] == "code"
assert isinstance(cell_info["semantic_types"], list)
assert "source_hash" not in cell_info
assert notebook_info["exported_py_hash"] == file_hash(export_path)
assert notebook_info["exported_py_path"].endswith("00_foundation_export.py")
assert cell_class_names(stamped_cell) == ("exported_code",)
stamped_cell.metadata["nbskill"]["semantic_types"] = ["stored_type"]
assert cell_class_names(stamped_cell) == ("stored_type",)
stamped_cell.source += chr(10)
assert cell_class_names(stamped_cell) == ("stored_type",)
remove_demo_path(export_path)